# Lunor App Studio

Most AI app builders take a prompt and hand you code. This one is my attempt at something different: it asks questions first, explains what it builds, checks its own work, and then turns the finished project into a learning path.

The flow: **idea → understand → plan → build → explain → verify and fix → learn.**

Run the cells from top to bottom.

**Good to know before you run it**

- You don't need an API key. Without one, the notebook builds a fixed expense tracker (SmartSpend) from templates. With an Anthropic or OpenAI key it builds whatever idea you give it. I haven't tested that path.
- The expense tracker ships with one bug on purpose: it accepts negative amounts. That gives the verify step something real to catch.
- Tests run in a subprocess on your machine. That is fine for a demo, but it is not a sandbox.
- Not included: deployment, vector search, multi-agent setups.

## 1. Setup

In [1]:
%pip install -q fastapi httpx pytest python-dotenv langgraph

Put your idea here. To build something other than the expense tracker, add an API key by uncommenting one line.

In [2]:
import os, getpass

# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

IDEA = ("Build an AI-powered expense tracking app for college students. Users should be able to record "
        "expenses, categorize them automatically, set monthly budgets, visualize spending patterns, and "
        "receive personalized AI recommendations to reduce unnecessary spending.")

## 2. The engine

The next cells write five small modules into a `lunor_app` folder, so the code stays readable and importable.

In [3]:
import os
os.makedirs('lunor_app', exist_ok=True)
open('lunor_app/__init__.py', 'w').close()

`state.py` holds everything about a project in one saved object. This is how the app remembers what it built.

In [4]:
%%writefile lunor_app/state.py
import json
import time
import uuid
from pathlib import Path

DATA = Path(__file__).parent / "data"
DATA.mkdir(exist_ok=True)

TASKS = [
    ("understand", "Requirements understood"),
    ("plan", "Development plan created"),
    ("architecture", "Architecture designed"),
    ("files", "Project files generated"),
    ("verify", "Verified (static checks + real tests)"),
    ("learn", "Learning path built from this project"),
]


def new_project(idea: str) -> dict:
    return {
        "id": uuid.uuid4().hex[:8],
        "idea": idea,
        "project_name": "",
        "mode": "",
        "orchestrator": "",
        "stage": "created",
        "error": "",
        "requirements": {},
        "questions": [],
        "answers": {},
        "plan": [],
        "architecture": {},
        "tasks": [{"id": i, "label": l, "status": "pending"} for i, l in TASKS],
        "files": {},
        "test_results": [],
        "issues": [],
        "health": None,
        "explanations": [],
        "learning_path": [],
        "learning_progress": {},
        "created": time.time(),
    }


def _path(pid: str) -> Path:
    if not pid.isalnum():
        raise KeyError(pid)
    return DATA / f"{pid}.json"


def save(p: dict) -> None:
    tmp = _path(p["id"]).with_suffix(".tmp")
    tmp.write_text(json.dumps(p, indent=2))
    tmp.replace(_path(p["id"]))


def load(pid: str) -> dict:
    path = _path(pid)
    if not path.exists():
        raise KeyError(pid)
    return json.loads(path.read_text())


def list_projects() -> list[dict]:
    out = []
    for f in sorted(DATA.glob("*.json"), key=lambda f: f.stat().st_mtime, reverse=True):
        try:
            p = json.loads(f.read_text())
            out.append({"id": p["id"], "name": p.get("project_name") or "Untitled",
                        "stage": p["stage"], "idea": p["idea"][:90]})
        except Exception:
            continue
    return out


def set_task(p: dict, tid: str, status: str) -> None:
    for t in p["tasks"]:
        if t["id"] == tid:
            t["status"] = status
    save(p)

Writing lunor_app/state.py


`llm.py` is a small client for Anthropic or OpenAI. It is only used when a key is set.

In [5]:
%%writefile lunor_app/llm.py
import json
import os

import httpx


def provider() -> str | None:
    if os.getenv("ANTHROPIC_API_KEY"):
        return "anthropic"
    if os.getenv("OPENAI_API_KEY"):
        return "openai"
    return None


def available() -> bool:
    return provider() is not None


def _extract_json(text: str) -> dict:
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("LLM did not return JSON")
    return json.loads(text[start:end + 1])


def ask_json(system: str, user: str, max_tokens: int = 6000) -> dict:
    system += "\nReply with ONE valid JSON object only. No markdown fences, no commentary."
    prov = provider()
    with httpx.Client(timeout=120) as c:
        if prov == "anthropic":
            r = c.post(
                "https://api.anthropic.com/v1/messages",
                headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"},
                json={"model": os.getenv("LLM_MODEL", "claude-sonnet-5-5"), "max_tokens": max_tokens,
                      "system": system, "messages": [{"role": "user", "content": user}]},
            )
            r.raise_for_status()
            return _extract_json(r.json()["content"][0]["text"])
        r = c.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
            json={"model": os.getenv("LLM_MODEL", "gpt-4o-mini"), "max_tokens": max_tokens,
                  "response_format": {"type": "json_object"},
                  "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]},
        )
        r.raise_for_status()
        return _extract_json(r.json()["choices"][0]["message"]["content"])

Writing lunor_app/llm.py


`blueprint.py` has the offline templates, the code explanations, the learning topics and the challenge.

In [6]:
%%writefile lunor_app/blueprint.py
import re


def understand(idea: str) -> dict:
    return {
        "project_name": "SmartSpend",
        "requirements": {
            "app": "SmartSpend",
            "target_users": "College students",
            "goal": "Help students track expenses and make better budget decisions.",
            "features": ["Expense tracking", "Automatic categorization", "Monthly budgets",
                         "Spending analytics", "AI recommendations"],
        },
        "questions": [
            {"id": "auth", "text": "How should users sign in?",
             "options": ["Email + Password", "Google", "No authentication"]},
            {"id": "db", "text": "Where should expense data be stored?",
             "options": ["PostgreSQL", "Firebase", "Local storage"]},
            {"id": "llm", "text": "Should AI recommendations use an LLM?", "options": ["Yes", "No"]},
        ],
    }


def plan(a: dict) -> list:
    steps = []
    if a.get("auth") != "No authentication":
        steps.append({"title": "Authentication", "items": ["Password hashing + signed tokens (helpers)", "Login / Signup (wiring is a learning exercise)"]})
    steps += [
        {"title": "Expense Management", "items": ["Add expense", "List expenses", "Edit expense"]},
        {"title": "Categorization", "items": ["Food", "Travel", "Education", "Entertainment"]},
        {"title": "Analytics", "items": ["Monthly spending", "Category breakdown", "Budget tracking"]},
        {"title": "AI Insights", "items": ["Spending analysis", "Budget recommendation"]},
    ]
    for i, s in enumerate(steps, 1):
        s["n"] = f"{i:02d}"
    return steps


def architecture(a: dict) -> dict:
    db = {"PostgreSQL": "PostgreSQL", "Firebase": "Firebase Firestore", "Local storage": "Browser localStorage"}[a.get("db", "PostgreSQL")]
    ai = "LLM + rule-based fallback" if a.get("llm") == "Yes" else "Rule-based engine"
    return {
        "layers": [
            {"name": "Frontend", "tech": "Next.js / React", "role": "Forms, dashboard and charts"},
            {"name": "API", "tech": "FastAPI", "role": "Validated JSON endpoints"},
            {"name": "Services", "tech": "Expenses · Auth · AI", "role": f"Business logic; AI = {ai}"},
            {"name": "Storage", "tech": db, "role": "Prototype keeps data in memory; schema/notes provided for the real store"},
        ],
        "why": ("React handles the interface, FastAPI gives a clean typed backend API, and the AI service "
                f"generates insights ({ai.lower()}). Storage is {db}; the generated prototype uses an in-memory "
                "store so it runs with zero setup."),
    }


_MODELS = '''from datetime import date
from typing import Literal, Optional

from pydantic import BaseModel

Category = Literal["Food", "Travel", "Education", "Entertainment", "Other"]


class ExpenseIn(BaseModel):
    title: str
    amount: float
    category: Optional[Category] = None
    spent_on: date
'''

_AI_BASE = '''KEYWORDS = {
    "Food": ["lunch", "dinner", "breakfast", "canteen", "cafe", "pizza", "snack", "grocer"],
    "Travel": ["bus", "metro", "uber", "ola", "train", "fuel", "flight", "auto"],
    "Education": ["book", "course", "tuition", "exam", "stationery", "notes"],
    "Entertainment": ["movie", "netflix", "game", "concert", "spotify"],
}


def categorize(title: str) -> str:
    text = title.lower()
    for category, words in KEYWORDS.items():
        if any(w in text for w in words):
            return category
    return "Other"


def recommend(expenses: list[dict], budget: float) -> list[str]:
    total = sum(e["amount"] for e in expenses)
    by_cat: dict[str, float] = {}
    for e in expenses:
        by_cat[e["category"]] = by_cat.get(e["category"], 0) + e["amount"]
    tips = []
    if total > budget:
        tips.append(f"You are over budget by {total - budget:.0f}. Pause non-essential spending.")
    elif total > 0.8 * budget:
        tips.append("You have used over 80% of your monthly budget.")
    if by_cat:
        top = max(by_cat, key=by_cat.get)
        tips.append(f"Your biggest category is {top} ({by_cat[top]:.0f}). Set a weekly cap for it.")
    return tips or ["Add a few expenses to get personalised tips."]
'''

_AI_LLM = '''

def llm_recommend(expenses: list[dict], budget: float) -> list[str]:
    """Ask an LLM for tips; fall back to the rule-based engine on any failure."""
    import os

    import httpx

    key = os.getenv("OPENAI_API_KEY")
    if not key:
        return recommend(expenses, budget)
    try:
        r = httpx.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content":
                  f"Budget {budget}. Expenses {expenses}. Give 3 short money-saving tips for a college student."}]},
            timeout=30,
        )
        r.raise_for_status()
        return [r.json()["choices"][0]["message"]["content"]]
    except Exception:
        return recommend(expenses, budget)
'''

_MAIN = '''from fastapi import FastAPI, HTTPException

from . import ai
from .models import ExpenseIn

app = FastAPI(title="SmartSpend API")

EXPENSES: dict[int, dict] = {}
BUDGET = {"monthly": 5000.0}
_next_id = 1


@app.post("/api/expenses")
def create_expense(data: ExpenseIn):
    global _next_id
    expense = data.model_dump(mode="json")
    expense["category"] = expense["category"] or ai.categorize(expense["title"])
    expense["id"] = _next_id
    EXPENSES[_next_id] = expense
    _next_id += 1
    return expense


@app.get("/api/expenses")
def list_expenses():
    return list(EXPENSES.values())


@app.put("/api/expenses/{expense_id}")
def update_expense(expense_id: int, data: ExpenseIn):
    if expense_id not in EXPENSES:
        raise HTTPException(status_code=404, detail="Expense not found")
    expense = data.model_dump(mode="json")
    expense["category"] = expense["category"] or ai.categorize(expense["title"])
    expense["id"] = expense_id
    EXPENSES[expense_id] = expense
    return expense


@app.get("/api/analytics")
def analytics():
    by_category: dict[str, float] = {}
    for e in EXPENSES.values():
        by_category[e["category"]] = by_category.get(e["category"], 0) + e["amount"]
    total = sum(by_category.values())
    return {"total": total, "budget": BUDGET["monthly"],
            "remaining": BUDGET["monthly"] - total, "by_category": by_category}


@app.get("/api/insights")
def insights():
    return {"tips": ai.recommend(list(EXPENSES.values()), BUDGET["monthly"])}
'''

_TESTS = '''from fastapi.testclient import TestClient

from backend.main import app

client = TestClient(app)


def _post(title="Lunch at canteen", amount=120):
    return client.post("/api/expenses", json={"title": title, "amount": amount, "spent_on": "2026-01-05"})


def test_create_expense_is_auto_categorized():
    r = _post()
    assert r.status_code == 200
    assert r.json()["category"] == "Food"


def test_list_and_analytics():
    _post("Metro ticket", 40)
    assert len(client.get("/api/expenses").json()) >= 1
    assert client.get("/api/analytics").json()["total"] > 0


def test_update_missing_expense_returns_404():
    r = client.put("/api/expenses/9999", json={"title": "x", "amount": 1, "spent_on": "2026-01-05"})
    assert r.status_code == 404


def test_rejects_negative_amount():
    assert _post("Refund?", -50).status_code == 422
'''

_AUTH_EMAIL = '''import base64
import hashlib
import hmac
import os

SECRET = os.getenv("APP_SECRET", "change-me").encode()


def hash_password(password: str, salt: bytes | None = None) -> str:
    salt = salt or os.urandom(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 200_000)
    return base64.b64encode(salt + digest).decode()


def verify_password(password: str, stored: str) -> bool:
    raw = base64.b64decode(stored)
    return hmac.compare_digest(hash_password(password, raw[:16]), stored)


def make_token(user_id: int) -> str:
    sig = hmac.new(SECRET, str(user_id).encode(), hashlib.sha256).hexdigest()
    return f"{user_id}.{sig}"
'''

_AUTH_GOOGLE = '''# Google sign-in: the browser obtains an ID token; the server only verifies it.
# pip install google-auth
from google.auth.transport import requests as g_requests
from google.oauth2 import id_token


def verify_google_token(token: str, client_id: str) -> dict:
    return id_token.verify_oauth2_token(token, g_requests.Request(), client_id)
'''

_SCHEMA = '''CREATE TABLE IF NOT EXISTS expenses (
    id         SERIAL PRIMARY KEY,
    title      TEXT NOT NULL,
    amount     NUMERIC(12, 2) NOT NULL CHECK (amount > 0),
    category   TEXT NOT NULL,
    spent_on   DATE NOT NULL
);

CREATE TABLE IF NOT EXISTS budgets (
    month      DATE PRIMARY KEY,
    limit_amt  NUMERIC(12, 2) NOT NULL
);
'''

_API_TS = '''export type NewExpense = { title: string; amount: number; spent_on: string };

export async function addExpense(expense: NewExpense) {
  const response = await fetch("/api/expenses", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify(expense),
  });
  if (!response.ok) throw new Error("Could not save expense");
  return response.json();
}

export async function listExpenses() {
  const response = await fetch("/api/expenses");
  return response.json();
}
'''

_PAGE_TSX = '''"use client";
import { useEffect, useState } from "react";
import { listExpenses } from "../../lib/api";

export default function Dashboard() {
  const [expenses, setExpenses] = useState<any[]>([]);

  useEffect(() => {
    listExpenses().then(setExpenses);
  }, []);

  const total = expenses.reduce((sum, e) => sum + e.amount, 0);
  return (
    <main>
      <h1>Total spent: {total}</h1>
      <ul>{expenses.map((e) => <li key={e.id}>{e.title} - {e.amount} ({e.category})</li>)}</ul>
    </main>
  );
}
'''


def files(a: dict) -> dict:
    f = {
        "backend/models.py": _MODELS,
        "backend/ai.py": _AI_BASE + (_AI_LLM if a.get("llm") == "Yes" else ""),
        "backend/main.py": _MAIN,
        "tests/test_expenses.py": _TESTS,
        "frontend/lib/api.ts": _API_TS,
        "frontend/app/dashboard/page.tsx": _PAGE_TSX,
        "README.md": ("# SmartSpend\n\nRun the API:\n\n```\npip install fastapi uvicorn httpx pytest\n"
                      "uvicorn backend.main:app --reload\npytest\n```\n"),
    }
    if a.get("auth") == "Email + Password":
        f["backend/auth.py"] = _AUTH_EMAIL
    elif a.get("auth") == "Google":
        f["backend/auth.py"] = _AUTH_GOOGLE
    if a.get("db") == "PostgreSQL":
        f["schema.sql"] = _SCHEMA
    return f


# pattern, what it does, analogy, why, alternative, topic
KB = [
    (r"fetch\(", "fetch() sends an HTTP request from the browser to our backend.",
     "Posting a letter to the bank and waiting for the reply.",
     "fetch is built into every browser, so no extra library is needed.",
     "axios, or TanStack Query for caching and retries.", "rest-apis"),
    (r"method:\s*[\"']POST|\.post\(|@app\.post", "POST tells the server we want to CREATE new data.",
     "Handing a new form to the clerk, as opposed to asking to read old forms.",
     "REST uses POST for creation, GET for reading, PUT for updating, DELETE for removing.",
     "PUT can also create when the client chooses the id.", "rest-apis"),
    (r"JSON\.stringify", "JSON.stringify() converts a JavaScript object into JSON text that can travel over HTTP.",
     "Packing items into a standard box so any courier can carry them.",
     "HTTP bodies are text, so objects must be serialized first.",
     "Send FormData for file uploads.", "react"),
    (r"\basync\b|\bawait\b", "async/await lets the code wait for slow work (network, disk) without freezing everything else.",
     "Ordering food and doing other things until it arrives.",
     "Reads like normal top-to-bottom code but never blocks.",
     "Promise .then() chains do the same with more nesting.", "react"),
    (r"@app\.(get|post|put|delete)", "A FastAPI decorator registers the function below it as the handler for that URL and HTTP method.",
     "A reception desk sign: 'expense forms go to window 3'.",
     "Decorators keep routing next to the logic; FastAPI also builds API docs from them automatically.",
     "Flask or Django REST framework use similar ideas with less typing.", "fastapi"),
    (r"BaseModel", "A Pydantic BaseModel declares the exact shape and types of incoming data and validates it automatically.",
     "A form template: wrong or missing fields are rejected at the door.",
     "Invalid input never reaches your business logic, and the error is a clear 422.",
     "dataclasses plus manual checks, or marshmallow.", "pydantic"),
    (r"Field\(", "Field(...) adds extra rules to one attribute, such as gt=0 meaning 'greater than zero'.",
     "A rule printed beside a form box: 'amount must be above zero'.",
     "Constraints live next to the field, so they cannot be forgotten in an endpoint.",
     "A @field_validator for rules that need custom logic.", "pydantic"),
    (r"HTTPException|raise\s", "raise HTTPException stops the request and returns an error with a proper status code.",
     "A clerk stamping 'REJECTED: not found' instead of silently losing the form.",
     "Clients can react to 404 or 422 correctly instead of guessing.",
     "Custom exception handlers with app.exception_handler.", "fastapi"),
    (r"def categorize", "categorize() scans the title for known keywords and returns the first matching category.",
     "A sorting clerk who checks for words like 'bus' or 'lunch'.",
     "Rules are fast, free, explainable and work offline.",
     "A trained text classifier or an LLM call for fuzzy cases.", "llm"),
    (r"useState|useEffect", "useState stores values that change; useEffect runs code after the component renders, such as loading data.",
     "useState is a notepad the component keeps; useEffect is a to-do that runs once the page is up.",
     "React re-renders automatically when state changes.",
     "Server components or SWR/React Query for data loading.", "react"),
    (r"CREATE TABLE", "CREATE TABLE defines a database table, its columns, types and constraints (CHECK, NOT NULL).",
     "Drawing the columns of a ledger before writing any rows.",
     "Constraints protect data even if application code has bugs.",
     "An ORM such as SQLAlchemy generates it from Python classes.", "database"),
    (r"pbkdf2_hmac", "PBKDF2 hashes the password with a random salt many times so stored passwords are hard to crack.",
     "Shredding and mixing a document so it can be checked but never read back.",
     "Never store plain passwords; slow hashing makes brute force expensive.",
     "bcrypt or argon2 via passlib.", "fastapi"),
    (r"TestClient|def test_", "TestClient calls the API in-process so tests run fast without starting a server.",
     "A mock customer testing the bank counter before opening day.",
     "Tests catch regressions, like a missing amount validation.",
     "Run against a staging server with httpx for end-to-end checks.", "fastapi"),
]


def explain(code: str) -> dict | None:
    hits = [row for row in KB if re.search(row[0], code)]
    if not hits:
        return None
    return {
        "summary": hits[0][1],
        "steps": [h[1] for h in hits[:6]],
        "analogy": hits[0][2],
        "why": " ".join(h[3] for h in hits[:3]),
        "alternative": " ".join(h[4] for h in hits[:3]),
        "topic": hits[0][5],
    }


def learning(a: dict) -> list:
    topics = [
        {"id": "react", "title": "React & the browser", "why": "The dashboard and api.ts use fetch, async/await and hooks.",
         "lesson": "The browser talks to your backend with fetch(). Data you send must be JSON text (JSON.stringify), and slow calls use async/await. React components keep changing data in state (useState) and load data after render (useEffect)."},
        {"id": "rest-apis", "title": "REST APIs", "why": "Your app exposes POST, GET and PUT endpoints.",
         "lesson": "REST maps actions to HTTP methods: POST creates, GET reads, PUT replaces, DELETE removes. URLs name things (/api/expenses/3), status codes report results (200 ok, 404 missing, 422 invalid input)."},
        {"id": "fastapi", "title": "FastAPI", "why": "backend/main.py is a FastAPI app.",
         "lesson": "FastAPI turns decorated Python functions into endpoints. Type hints become validation and automatic docs at /docs. Raise HTTPException to return clean errors. Use TestClient to test without a server."},
        {"id": "pydantic", "title": "Validation with Pydantic", "why": "ExpenseIn defines what a valid expense is.",
         "lesson": "A BaseModel describes allowed data. Field(gt=0) adds constraints. Invalid payloads are rejected with HTTP 422 before your code runs, which is why the missing amount rule was a real bug."},
    ]
    if a.get("db") == "PostgreSQL":
        topics.append({"id": "database", "title": "PostgreSQL", "why": "schema.sql defines your tables.",
                       "lesson": "Tables hold rows with typed columns. Constraints (NOT NULL, CHECK) protect data integrity. SERIAL gives automatic ids. Indexes speed up lookups on columns you filter by."})
    if a.get("llm") == "Yes":
        topics.append({"id": "llm", "title": "LLM integration", "why": "ai.py can call an LLM and falls back to rules.",
                       "lesson": "Send the model a focused prompt with your data, ask for a small structured answer, and always keep a fallback for timeouts or errors. Keep API keys in environment variables, never in code."})
    else:
        topics.append({"id": "llm", "title": "Rule-based AI", "why": "ai.py categorizes with keyword rules.",
                       "lesson": "Simple rules are fast, free and explainable. When rules fail on fuzzy input, that is the signal to try a classifier or an LLM."})
    return topics


CHALLENGE = {
    "topic": "rest-apis",
    "title": "Build DELETE /api/expenses/{id}",
    "brief": "Add a delete endpoint to backend/main.py. It must return 404 for unknown ids and {\"deleted\": id} otherwise. The store is the EXPENSES dict.",
    "starter": '''@app.delete("/api/expenses/{expense_id}")
def delete_expense(expense_id: int):
    # TODO: raise HTTPException(404) if the id is unknown,
    # otherwise remove it from EXPENSES and return {"deleted": expense_id}
    ...
''',
    "test": '''from fastapi.testclient import TestClient

from backend.main import app

c = TestClient(app)


def test_delete_flow():
    r = c.post("/api/expenses", json={"title": "Bus", "amount": 20, "spent_on": "2026-01-05"})
    eid = r.json()["id"]
    d = c.delete(f"/api/expenses/{eid}")
    assert d.status_code == 200 and d.json() == {"deleted": eid}
    assert c.delete(f"/api/expenses/{eid}").status_code == 404
''',
}

Writing lunor_app/blueprint.py


`verify.py` runs the checks and the real tests, applies fixes, and grades the challenge.

In [7]:
%%writefile lunor_app/verify.py
import ast
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

VALIDATION_RE = re.compile(r"\bgt\s*=\s*0(?![.\d])|\bge\s*=\s*0?\.0*1|amount\w*\s*(<=|<)\s*0(?![.\d])|amount\w*\s*>\s*0(?![.\d])")


def _write(files: dict, root: Path) -> None:
    for name, content in files.items():
        dest = (root / name).resolve()
        if root.resolve() not in dest.parents:
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(content)
    init = root / "backend" / "__init__.py"
    if (root / "backend").exists() and not init.exists():
        init.write_text("")


def run_pytest(files: dict, args: list[str] | None = None) -> dict:
    if os.getenv("LUNOR_RUN_TESTS", "1") == "0":
        return {"ok": None, "passed": 0, "failed": 0, "failures": [], "output": "Test execution disabled."}
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        _write(files, root)
        try:
            # runs generated tests in a subprocess: fine for a demo, not a sandbox
            r = subprocess.run(
                [sys.executable, "-m", "pytest", "-q", "-rf", "--tb=short", "-p", "no:cacheprovider"] + (args or []),
                cwd=root, env={**os.environ, "PYTHONPATH": str(root)},
                capture_output=True, text=True, timeout=60,
            )
        except subprocess.TimeoutExpired:
            return {"ok": False, "passed": 0, "failed": 1, "failures": ["timeout"], "output": "Tests timed out."}
    out = (r.stdout + r.stderr)[-2500:]
    passed = int((re.search(r"(\d+) passed", out) or [0, 0])[1])
    failed = int((re.search(r"(\d+) failed", out) or [0, 0])[1])
    errors = int((re.search(r"(\d+) error", out) or [0, 0])[1])
    return {"ok": r.returncode == 0, "passed": passed, "failed": failed + errors,
            "failures": re.findall(r"FAILED (\S+)", out), "output": out}


def static_checks(files: dict) -> list[dict]:
    py = {k: v for k, v in files.items() if k.endswith(".py")}
    checks = []

    bad = []
    for name, src in py.items():
        try:
            ast.parse(src)
        except SyntaxError as e:
            bad.append(f"{name}:{e.lineno}")
    checks.append({"id": "syntax", "name": "Python files parse", "ok": not bad,
                   "detail": "All files parse." if not bad else "Syntax errors: " + ", ".join(bad)})

    main = files.get("backend/main.py", "")
    checks.append({"id": "structure", "name": "API entrypoint (FastAPI app)", "ok": "FastAPI(" in main,
                   "detail": "backend/main.py creates a FastAPI app." if "FastAPI(" in main else "backend/main.py missing or has no FastAPI app."})

    routes = len(re.findall(r"@app\.(get|post|put|patch|delete)\(", main))
    checks.append({"id": "routes", "name": "At least 3 API routes", "ok": routes >= 3, "detail": f"{routes} routes found."})

    has_exc = any("HTTPException" in v for v in py.values())
    checks.append({"id": "errors", "name": "Error handling (HTTPException)", "ok": has_exc,
                   "detail": "HTTPException is used." if has_exc else "No HTTPException found."})

    mentions_amount = any("amount" in v for v in py.values())
    validated = any(VALIDATION_RE.search(v) for k, v in py.items() if not k.startswith("tests/"))
    if mentions_amount:
        checks.append({"id": "amount", "name": "Amounts are validated as positive", "ok": validated,
                       "detail": "A positive-amount rule exists." if validated else "No positive-amount validation found; negative expenses are accepted."})

    has_tests = any(k.startswith("tests/") for k in py)
    checks.append({"id": "tests", "name": "Tests exist", "ok": has_tests,
                   "detail": "Test files found." if has_tests else "No tests/ directory."})
    return checks


def run(p: dict) -> None:
    files = p["files"]
    checks = static_checks(files)
    tests = run_pytest(files) if any(k.startswith("tests/") for k in files) else None
    p["test_results"] = checks + ([{"id": "pytest", "name": "pytest (real run)", "ok": tests["ok"],
                                    "detail": f"{tests['passed']} passed, {tests['failed']} failed",
                                    "output": tests["output"]}] if tests else [])

    issues = []
    for c in checks:
        if not c["ok"]:
            issues.append({"id": c["id"], "severity": "Medium" if c["id"] in ("amount", "errors") else "Low",
                           "title": c["name"] + " failed", "detail": c["detail"], "fixable": c["id"] == "amount"})
    if tests and tests["failed"]:
        names = ", ".join(tests["failures"]) or "see output"
        issues.append({"id": "pytest", "severity": "High" if tests["failed"] > 1 else "Medium",
                       "title": f"{tests['failed']} test(s) failing", "detail": names,
                       "fixable": any("negative" in n for n in tests["failures"]) and not validated(files)})

    total = len(checks) + (tests["passed"] + tests["failed"] if tests else 0)
    good = sum(c["ok"] for c in checks) + (tests["passed"] if tests else 0)
    p["issues"] = issues
    p["health"] = round(100 * good / total) if total else 0


def validated(files: dict) -> bool:
    return any(VALIDATION_RE.search(v) for k, v in files.items() if k.endswith(".py") and not k.startswith("tests/"))


def apply_fix(p: dict, issue_id: str) -> dict:
    if issue_id not in ("amount", "pytest"):
        raise ValueError("This issue has no automatic fix.")
    for name, src in p["files"].items():
        if not name.endswith(".py") or "BaseModel" not in src:
            continue
        m = re.search(r"^(\s*)amount:\s*float\b(?!\s*=)", src, re.M)
        if not m:
            continue
        before = m.group(0).strip()
        after = before + ' = Field(gt=0, description="Must be positive")'
        new = src[:m.start()] + m.group(1) + after + src[m.end():]
        if not re.search(r"from pydantic import[^\n]*\bField\b", new):
            new = re.sub(r"from pydantic import ([^\n]+)", lambda x: "from pydantic import " + x.group(1) + ", Field", new, count=1)
        p["files"][name] = new
        return {"file": name, "before": before, "after": after}
    raise ValueError("Could not find an `amount: float` field to patch automatically.")


def check_challenge(files: dict, code: str, test_src: str) -> dict:
    if len(code) > 5000:
        return {"ok": False, "output": "Submission too long."}
    f = dict(files)
    f["backend/main.py"] = f["backend/main.py"].rstrip() + "\n\n\n" + code + "\n"
    f["tests/test_challenge.py"] = test_src
    f = {k: v for k, v in f.items() if not k.startswith("tests/") or k == "tests/test_challenge.py"}
    res = run_pytest(f)
    return {"ok": bool(res["ok"]), "output": res["output"]}

Writing lunor_app/verify.py


`graph.py` is the orchestrator. Each stage is a node in a LangGraph graph that reads and updates the project state.

In [8]:
%%writefile lunor_app/graph.py
from typing import TypedDict

from . import blueprint, llm, state, verify

try:
    from langgraph.graph import END, START, StateGraph
    HAVE_LANGGRAPH = True
except ImportError:
    HAVE_LANGGRAPH = False


class S(TypedDict, total=False):
    project: dict


def _ctx(p: dict) -> str:
    return (f"Idea: {p['idea']}\nRequirements: {p['requirements']}\nUser decisions: {p['answers']}\n"
            f"Architecture: {p['architecture']}")


def n_understand(p: dict) -> None:
    if llm.available():
        r = llm.ask_json(
            "You are a product analyst. Extract requirements and ask exactly 3 clarification questions "
            "with 2-3 options each. Schema: {\"project_name\":str,\"requirements\":{\"app\":str,\"target_users\":str,"
            "\"goal\":str,\"features\":[str]},\"questions\":[{\"id\":str,\"text\":str,\"options\":[str]}]}",
            p["idea"])
        p["mode"] = "llm"
    else:
        r = blueprint.understand(p["idea"])
        p["mode"] = "offline-template"
    p["project_name"], p["requirements"], p["questions"] = r["project_name"], r["requirements"], r["questions"]
    p["stage"] = "clarifying"


def n_plan(p: dict) -> None:
    if p["mode"] == "llm":
        r = llm.ask_json("Create a development plan. Schema: {\"plan\":[{\"n\":\"01\",\"title\":str,\"items\":[str]}]} (4-6 modules).", _ctx(p))
        p["plan"] = r["plan"]
    else:
        p["plan"] = blueprint.plan(p["answers"])


def n_architect(p: dict) -> None:
    if p["mode"] == "llm":
        p["architecture"] = llm.ask_json(
            "Design the architecture honoring the user's decisions. Schema: {\"layers\":[{\"name\":str,\"tech\":str,\"role\":str}],"
            "\"why\":str} (3-5 layers, top to bottom; 'why' explains each choice in plain language).", _ctx(p))
    else:
        p["architecture"] = blueprint.architecture(p["answers"])


def n_build(p: dict) -> None:
    if p["mode"] == "llm":
        r = llm.ask_json(
            "Generate a SMALL but runnable project (max 8 files, each under 120 lines). Use FastAPI backend in backend/ "
            "(with relative imports, backend/main.py exposing `app`), pytest tests in tests/ importing `from backend.main import app`, "
            "plus minimal frontend files. Keep the store in memory. Schema: {\"files\":{\"path\":\"content\"}}", _ctx(p), max_tokens=12000)
        p["files"] = {k: v for k, v in r["files"].items() if isinstance(v, str) and ".." not in k and not k.startswith("/")}
    else:
        p["files"] = blueprint.files(p["answers"])


def n_verify(p: dict) -> None:
    verify.run(p)


def n_learn(p: dict) -> None:
    if p["mode"] == "llm":
        r = llm.ask_json(
            "Build a learning path from the technologies actually used in this project. Schema: {\"topics\":[{\"id\":str,\"title\":str,"
            "\"why\":str,\"lesson\":str}]} (4-7 topics, lesson = 3-5 sentences).", _ctx(p) + f"\nFiles: {list(p['files'])}")
        p["learning_path"] = r["topics"]
    else:
        p["learning_path"] = blueprint.learning(p["answers"])
    p["learning_progress"] = {t["id"]: {"read": False, "explained": 0, "challenge": False} for t in p["learning_path"]}


BUILD_NODES = [("plan", n_plan, "plan"), ("architect", n_architect, "architecture"),
               ("build", n_build, "files"), ("verify", n_verify, "verify"), ("learn", n_learn, "learn")]


def _wrap(fn, tid):
    def run(s: S) -> S:
        p = s["project"]
        if tid:
            state.set_task(p, tid, "running")
        fn(p)
        if tid:
            state.set_task(p, tid, "done")
        state.save(p)
        return {"project": p}
    return run


def _run(nodes, p: dict) -> None:
    if HAVE_LANGGRAPH:
        g = StateGraph(S)
        prev = START
        for nid, fn, tid in nodes:
            g.add_node(nid, _wrap(fn, tid))
            g.add_edge(prev, nid)
            prev = nid
        g.add_edge(prev, END)
        g.compile().invoke({"project": p})
        p["orchestrator"] = "langgraph"
    else:
        for _, fn, tid in nodes:
            _wrap(fn, tid)({"project": p})
        p["orchestrator"] = "sequential"


def start(p: dict) -> None:
    _run([("understand", n_understand, "understand")], p)
    state.save(p)


def build(pid: str) -> None:
    p = state.load(pid)
    try:
        _run(BUILD_NODES, p)
        p["stage"] = "ready"
    except Exception as e:
        p["stage"], p["error"] = "error", f"{type(e).__name__}: {e}"
        for t in p["tasks"]:
            if t["status"] == "running":
                t["status"] = "failed"
    state.save(p)


def explain(p: dict, code: str) -> dict:
    if p["mode"] == "llm":
        r = llm.ask_json(
            "Explain this code to a learner. Schema: {\"summary\":str,\"steps\":[str],\"analogy\":str,\"why\":str,\"alternative\":str,"
            f"\"topic\":one of {[t['id'] for t in p['learning_path']]}}}", f"Project: {p['project_name']}\nCode:\n{code}")
    else:
        r = blueprint.explain(code)
        if r is None:
            raise ValueError("Offline mode only recognises the patterns in the generated demo code. Set OPENAI_API_KEY or ANTHROPIC_API_KEY to explain any code.")
    prog = p["learning_progress"].get(r.get("topic"))
    if prog:
        prog["explained"] += 1
    p["explanations"].append({"code": code[:400], **r})
    state.save(p)
    return r

Writing lunor_app/graph.py


Now import the modules and add a few helpers for printing.

In [9]:
import importlib, json, shutil, pathlib
importlib.invalidate_caches()
from IPython.display import display, Markdown

from lunor_app import state, llm, blueprint, verify, graph

def md(text):
    display(Markdown(text))

def bar(pct, width=20):
    filled = round(width * pct / 100)
    return "█" * filled + "░" * (width - filled) + f" {pct}%"

def show_tasks(p):
    icons = {"done": "✓", "running": "...", "failed": "✗", "pending": "○"}
    md("**Build pipeline**\n\n" + "\n".join(f"- {icons[t['status']]} {t['label']}" for t in p["tasks"]))

def show_file(p, name):
    lang = {"py": "python", "ts": "typescript", "tsx": "tsx", "sql": "sql", "md": "markdown"}.get(name.rsplit(".", 1)[-1], "")
    md(f"**{name}**\n\n```{lang}\n{p['files'][name]}\n```")

mode = "LLM (" + llm.provider() + ")" if llm.available() else "offline templates (SmartSpend demo)"
print("Mode:", mode, "| LangGraph installed:", graph.HAVE_LANGGRAPH)

Mode: offline templates (SmartSpend demo) | LangGraph installed: True


## 3. Understand

First it reads the idea and says back what it understood.

In [10]:
p = state.new_project(IDEA)
graph.start(p)
r = p["requirements"]
md(f"**App:** {r['app']}  \n**Target users:** {r['target_users']}  \n**Goal:** {r['goal']}\n\n**Core features**\n\n"
   + "\n".join(f"- {f}" for f in r["features"]))

**App:** SmartSpend  
**Target users:** College students  
**Goal:** Help students track expenses and make better budget decisions.

**Core features**

- Expense tracking
- Automatic categorization
- Monthly budgets
- Spending analytics
- AI recommendations

Then it asks the questions that change how the app gets built, instead of guessing.

In [11]:
md("\n\n".join(
    f"**{i}. {q['text']}**  \n" + "  \n".join(f"○ {o}" for o in q["options"])
    for i, q in enumerate(p["questions"], 1)))

**1. How should users sign in?**  
○ Email + Password  
○ Google  
○ No authentication

**2. Where should expense data be stored?**  
○ PostgreSQL  
○ Firebase  
○ Local storage

**3. Should AI recommendations use an LLM?**  
○ Yes  
○ No

Pick your answers. The defaults are the first option of each question; uncomment a line to change one.

In [12]:
ANSWERS = {q["id"]: q["options"][0] for q in p["questions"]}
ANSWERS.update({
    # "auth": "Google",
    # "db": "Firebase",
    # "llm": "No",
})
for q in p["questions"]:
    assert ANSWERS[q["id"]] in q["options"], f"Invalid choice for {q['id']}: {ANSWERS[q['id']]}"
    print(f"{q['text']:<45} → {ANSWERS[q['id']]}")

How should users sign in?                     → Email + Password
Where should expense data be stored?          → PostgreSQL
Should AI recommendations use an LLM?         → Yes


## 4. Plan and build

This one call runs plan, architecture, code generation, verification and the learning path through the LangGraph graph.

In [13]:
p["answers"] = ANSWERS
p["stage"] = "building"
state.save(p)
graph.build(p["id"])
p = state.load(p["id"])
print("Orchestrator:", p["orchestrator"], "| stage:", p["stage"], p["error"] or "")
show_tasks(p)

Orchestrator: langgraph | stage: ready 


**Build pipeline**

- ✓ Requirements understood
- ✓ Development plan created
- ✓ Architecture designed
- ✓ Project files generated
- ✓ Verified (static checks + real tests)
- ✓ Learning path built from this project

The development plan:

In [14]:
md("\n".join(f"**{m['n']}  {m['title']}**\n" + "\n".join(f"  - {it}" for it in m["items"]) + "\n" for m in p["plan"]))

**01  Authentication**
  - Password hashing + signed tokens (helpers)
  - Login / Signup (wiring is a learning exercise)

**02  Expense Management**
  - Add expense
  - List expenses
  - Edit expense

**03  Categorization**
  - Food
  - Travel
  - Education
  - Entertainment

**04  Analytics**
  - Monthly spending
  - Category breakdown
  - Budget tracking

**05  AI Insights**
  - Spending analysis
  - Budget recommendation


The architecture, and why it was chosen:

In [15]:
a = p["architecture"]
w = 76
blocks = []
for i, l in enumerate(a["layers"]):
    if i:
        blocks.append(" " * (w // 2) + "│\n" + " " * (w // 2) + "▼")
    blocks.append("┌" + "─" * w + "┐\n│" + f"{l['name']} · {l['tech']}".center(w) + "│\n│" + l["role"][:w].center(w) + "│\n└" + "─" * w + "┘")
print("\n".join(blocks))
md("**Why this architecture?**\n\n" + a["why"])

┌────────────────────────────────────────────────────────────────────────────┐
│                         Frontend · Next.js / React                         │
│                        Forms, dashboard and charts                         │
└────────────────────────────────────────────────────────────────────────────┘
                                      │
                                      ▼
┌────────────────────────────────────────────────────────────────────────────┐
│                               API · FastAPI                                │
│                          Validated JSON endpoints                          │
└────────────────────────────────────────────────────────────────────────────┘
                                      │
                                      ▼
┌────────────────────────────────────────────────────────────────────────────┐
│                      Services · Expenses · Auth · AI                       │
│               Business logic; AI = LLM + rule-ba

**Why this architecture?**

React handles the interface, FastAPI gives a clean typed backend API, and the AI service generates insights (llm + rule-based fallback). Storage is PostgreSQL; the generated prototype uses an in-memory store so it runs with zero setup.

The generated files:

In [16]:
print(p["project_name"])
for name in sorted(p["files"]):
    print("  ", name, f"({len(p['files'][name].splitlines())} lines)")
show_file(p, "backend/models.py")
show_file(p, "backend/main.py")

SmartSpend
   README.md (9 lines)
   backend/ai.py (53 lines)
   backend/auth.py (22 lines)
   backend/main.py (52 lines)
   backend/models.py (13 lines)
   frontend/app/dashboard/page.tsx (19 lines)
   frontend/lib/api.ts (16 lines)
   schema.sql (12 lines)
   tests/test_expenses.py (30 lines)


**backend/models.py**

```python
from datetime import date
from typing import Literal, Optional

from pydantic import BaseModel

Category = Literal["Food", "Travel", "Education", "Entertainment", "Other"]


class ExpenseIn(BaseModel):
    title: str
    amount: float
    category: Optional[Category] = None
    spent_on: date

```

**backend/main.py**

```python
from fastapi import FastAPI, HTTPException

from . import ai
from .models import ExpenseIn

app = FastAPI(title="SmartSpend API")

EXPENSES: dict[int, dict] = {}
BUDGET = {"monthly": 5000.0}
_next_id = 1


@app.post("/api/expenses")
def create_expense(data: ExpenseIn):
    global _next_id
    expense = data.model_dump(mode="json")
    expense["category"] = expense["category"] or ai.categorize(expense["title"])
    expense["id"] = _next_id
    EXPENSES[_next_id] = expense
    _next_id += 1
    return expense


@app.get("/api/expenses")
def list_expenses():
    return list(EXPENSES.values())


@app.put("/api/expenses/{expense_id}")
def update_expense(expense_id: int, data: ExpenseIn):
    if expense_id not in EXPENSES:
        raise HTTPException(status_code=404, detail="Expense not found")
    expense = data.model_dump(mode="json")
    expense["category"] = expense["category"] or ai.categorize(expense["title"])
    expense["id"] = expense_id
    EXPENSES[expense_id] = expense
    return expense


@app.get("/api/analytics")
def analytics():
    by_category: dict[str, float] = {}
    for e in EXPENSES.values():
        by_category[e["category"]] = by_category.get(e["category"], 0) + e["amount"]
    total = sum(by_category.values())
    return {"total": total, "budget": BUDGET["monthly"],
            "remaining": BUDGET["monthly"] - total, "by_category": by_category}


@app.get("/api/insights")
def insights():
    return {"tips": ai.recommend(list(EXPENSES.values()), BUDGET["monthly"])}

```

Change the file name to read any other generated file.

In [17]:
show_file(p, "frontend/lib/api.ts")

**frontend/lib/api.ts**

```typescript
export type NewExpense = { title: string; amount: number; spent_on: string };

export async function addExpense(expense: NewExpense) {
  const response = await fetch("/api/expenses", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify(expense),
  });
  if (!response.ok) throw new Error("Could not save expense");
  return response.json();
}

export async function listExpenses() {
  const response = await fetch("/api/expenses");
  return response.json();
}

```

## 5. Explain

Paste any snippet from the files above into `CODE_TO_EXPLAIN`. Offline, it recognises the patterns used in the demo code; with an API key it can explain anything.

In [18]:
CODE_TO_EXPLAIN = """const response = await fetch("/api/expenses", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify(expense),
});"""

e = graph.explain(p, CODE_TO_EXPLAIN)
md("**What this code does**\n\n" + e["summary"])
md("\n\n".join(f"**Step {i}**  \n{s}" for i, s in enumerate(e["steps"], 1)))
md("**Think of it like this:** " + e["analogy"])

**What this code does**

fetch() sends an HTTP request from the browser to our backend.

**Step 1**  
fetch() sends an HTTP request from the browser to our backend.

**Step 2**  
POST tells the server we want to CREATE new data.

**Step 3**  
JSON.stringify() converts a JavaScript object into JSON text that can travel over HTTP.

**Step 4**  
async/await lets the code wait for slow work (network, disk) without freezing everything else.

**Think of it like this:** Posting a letter to the bank and waiting for the reply.

And the follow-up questions you might ask next: why this way, and what else could work.

In [19]:
md("**Why this approach?**  \n" + e["why"])
md("**Alternative:**  \n" + e["alternative"])

**Why this approach?**  
fetch is built into every browser, so no extra library is needed. REST uses POST for creation, GET for reading, PUT for updating, DELETE for removing. HTTP bodies are text, so objects must be serialized first.

**Alternative:**  
axios, or TanStack Query for caching and retries. PUT can also create when the client chooses the id. Send FormData for file uploads.

## 6. Verify and fix

The reviewer runs structure checks and the generated project's own tests. Nothing here is simulated.

In [20]:
md(f"**Project health: {p['health']}%**")
for c in p["test_results"]:
    print("✓" if c["ok"] else "-" if c["ok"] is None else "✗", c["name"], "|", c["detail"])
print()
for i in p["issues"]:
    print(f"[{i['severity']}] {i['title']}: {i['detail']} (auto-fix: {i['fixable']})")
if not p["issues"]:
    print("No issues found.")

**Project health: 80%**

✓ Python files parse | All files parse.
✓ API entrypoint (FastAPI app) | backend/main.py creates a FastAPI app.
✓ At least 3 API routes | 5 routes found.
✓ Error handling (HTTPException) | HTTPException is used.
✗ Amounts are validated as positive | No positive-amount validation found; negative expenses are accepted.
✓ Tests exist | Test files found.
✗ pytest (real run) | 3 passed, 1 failed

[Medium] Amounts are validated as positive failed: No positive-amount validation found; negative expenses are accepted. (auto-fix: True)
[Medium] 1 test(s) failing: see output (auto-fix: False)


The raw pytest output, in case you want to see what failed:

In [21]:
pt = next((c for c in p["test_results"] if c["id"] == "pytest"), None)
print(pt["output"] if pt else "No tests were run.")

...F                                                                     [100%]
=================================== FAILURES ===================================
_________________________ test_rejects_negative_amount _________________________
tests/test_expenses.py:30: in test_rejects_negative_amount
    assert _post("Refund?", -50).status_code == 422
E   AssertionError: assert 200 == 422
E    +  where 200 = <Response [200 OK]>.status_code
E    +    where <Response [200 OK]> = _post('Refund?', -50)
=============================== warnings summary ===============================
../../usr/local/lib/python3.13/dist-packages/starlette/testclient.py:53
  /usr/local/lib/python3.13/dist-packages/starlette/testclient.py:53: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.ht

Now apply the fix and verify again. The tests run a second time, so the new health score is earned.

In [22]:
before = p["health"]
fixable = [i for i in p["issues"] if i["fixable"]]
if not fixable:
    print("Nothing auto-fixable right now.")
else:
    change = verify.apply_fix(p, fixable[0]["id"])
    verify.run(p)
    state.save(p)
    md(f"**Fix applied** in `{change['file']}`\n\n```diff\n- {change['before']}\n+ {change['after']}\n```")
    print(f"Health {before}% → {p['health']}%")
    print("Remaining issues:", [i["title"] for i in p["issues"]] or "none")

**Fix applied** in `backend/models.py`

```diff
- amount: float
+ amount: float = Field(gt=0, description="Must be positive")
```

Health 80% → 100%
Remaining issues: none


## 7. Learn

The learning path comes from what was actually built. Progress is earned: reading a lesson gives 40%, each piece of code you explain from that topic gives 20% (up to three), and passing the challenge counts as one of those.

In [23]:
def pct(t):
    g = p["learning_progress"].get(t["id"], {})
    return min(100, (40 if g.get("read") else 0) + 20 * min(3, g.get("explained", 0) + (1 if g.get("challenge") else 0)))

def show_learning():
    md(f"**Your {p['project_name']} learning path**")
    for i, t in enumerate(p["learning_path"], 1):
        print(f"{i:02d}  {t['title']:<30}{bar(pct(t))}")

show_learning()

**Your SmartSpend learning path**

01  React & the browser           ░░░░░░░░░░░░░░░░░░░░ 0%
02  REST APIs                     ████░░░░░░░░░░░░░░░░ 20%
03  FastAPI                       ░░░░░░░░░░░░░░░░░░░░ 0%
04  Validation with Pydantic      ░░░░░░░░░░░░░░░░░░░░ 0%
05  PostgreSQL                    ░░░░░░░░░░░░░░░░░░░░ 0%
06  LLM integration               ░░░░░░░░░░░░░░░░░░░░ 0%


Read a lesson. Pick any topic id from the list.

In [24]:
print("Topics:", [t["id"] for t in p["learning_path"]])
TOPIC = p["learning_path"][0]["id"]
t = next(t for t in p["learning_path"] if t["id"] == TOPIC)
md(f"**{t['title']}**\n\n*Why it matters here:* {t['why']}\n\n{t['lesson']}")
p["learning_progress"][TOPIC]["read"] = True
state.save(p)
show_learning()

Topics: ['react', 'rest-apis', 'fastapi', 'pydantic', 'database', 'llm']


**React & the browser**

*Why it matters here:* The dashboard and api.ts use fetch, async/await and hooks.

The browser talks to your backend with fetch(). Data you send must be JSON text (JSON.stringify), and slow calls use async/await. React components keep changing data in state (useState) and load data after render (useEffect).

**Your SmartSpend learning path**

01  React & the browser           ████████░░░░░░░░░░░░ 40%
02  REST APIs                     ████░░░░░░░░░░░░░░░░ 20%
03  FastAPI                       ░░░░░░░░░░░░░░░░░░░░ 0%
04  Validation with Pydantic      ░░░░░░░░░░░░░░░░░░░░ 0%
05  PostgreSQL                    ░░░░░░░░░░░░░░░░░░░░ 0%
06  LLM integration               ░░░░░░░░░░░░░░░░░░░░ 0%


Now a challenge: build the `DELETE /api/expenses/{id}` endpoint yourself.

In [25]:
OFFLINE = p["mode"] == "offline-template"
if OFFLINE:
    ch = blueprint.CHALLENGE
    md(f"**{ch['title']}**\n\n{ch['brief']}\n\n```python\n{ch['starter']}```")
else:
    print("Challenges are only available for the built-in SmartSpend project.")

**Build DELETE /api/expenses/{id}**

Add a delete endpoint to backend/main.py. It must return 404 for unknown ids and {"deleted": id} otherwise. The store is the EXPENSES dict.

```python
@app.delete("/api/expenses/{expense_id}")
def delete_expense(expense_id: int):
    # TODO: raise HTTPException(404) if the id is unknown,
    # otherwise remove it from EXPENSES and return {"deleted": expense_id}
    ...
```

Write your solution and run the cell. A real test creates an expense, deletes it, then tries to delete it again.

In [26]:
MY_SOLUTION = """@app.delete("/api/expenses/{expense_id}")
def delete_expense(expense_id: int):
    ...
"""

if OFFLINE:
    res = verify.check_challenge(p["files"], MY_SOLUTION, blueprint.CHALLENGE["test"])
    print("Passed" if res["ok"] else "Not yet")
    print(res["output"][-900:])
    if res["ok"]:
        p["learning_progress"][blueprint.CHALLENGE["topic"]]["challenge"] = True
        state.save(p)

Not yet
E    +  and   None = json()
E    +    where json = <Response [200 OK]>.json
=============================== warnings summary ===============================
../../usr/local/lib/python3.13/dist-packages/starlette/testclient.py:53
  /usr/local/lib/python3.13/dist-packages/starlette/testclient.py:53: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
=========================== short test summary info ============================
FAILED tests/test_challenge.py::test_delete_flow - AssertionError: assert (200 == 200 and None == {'deleted': 1})
1 failed, 1 warning in 0.22s



If you get stuck, here is one working solution. Try it yourself first.

In [27]:
REFERENCE = """@app.delete("/api/expenses/{expense_id}")
def delete_expense(expense_id: int):
    if expense_id not in EXPENSES:
        raise HTTPException(status_code=404, detail="Expense not found")
    del EXPENSES[expense_id]
    return {"deleted": expense_id}
"""

if OFFLINE:
    res = verify.check_challenge(p["files"], REFERENCE, blueprint.CHALLENGE["test"])
    print("Passed" if res["ok"] else "Not yet")
    if res["ok"]:
        p["learning_progress"][blueprint.CHALLENGE["topic"]]["challenge"] = True
        state.save(p)
    show_learning()

Passed


**Your SmartSpend learning path**

01  React & the browser           ████████░░░░░░░░░░░░ 40%
02  REST APIs                     ████████░░░░░░░░░░░░ 40%
03  FastAPI                       ░░░░░░░░░░░░░░░░░░░░ 0%
04  Validation with Pydantic      ░░░░░░░░░░░░░░░░░░░░ 0%
05  PostgreSQL                    ░░░░░░░░░░░░░░░░░░░░ 0%
06  LLM integration               ░░░░░░░░░░░░░░░░░░░░ 0%


## 8. Saved state and export

Everything above lives in one JSON file. A later action would start from this file, not from a blank prompt.

In [28]:
import os
path = state.DATA / f"{p['id']}.json"
saved = json.loads(path.read_text())
print("Saved at:", os.path.relpath(path))
for k, v in saved.items():
    size = f"[{len(v)}]" if isinstance(v, (list, dict, str)) else ""
    print(f"  {k:<18}{type(v).__name__}{size}")

again = state.load(p["id"])
print(f"\nReloaded: {again['project_name']}, {len(again['files'])} files, {len(again['explanations'])} explanation(s), health {again['health']}%")

Saved at: lunor_app/data/6bc431b2.json
  id                str[8]
  idea              str[260]
  project_name      str[10]
  mode              str[16]
  orchestrator      str[9]
  stage             str[5]
  error             str[0]
  requirements      dict[4]
  questions         list[3]
  answers           dict[3]
  plan              list[5]
  architecture      dict[2]
  tasks             list[6]
  files             dict[9]
  test_results      list[7]
  issues            list[0]
  health            int
  explanations      list[1]
  learning_path     list[6]
  learning_progress dict[6]
  created           float

Reloaded: SmartSpend, 9 files, 1 explanation(s), health 100%


Export the generated project as a zip.

In [29]:
out = pathlib.Path("generated_project") / p["project_name"]
shutil.rmtree(out.parent, ignore_errors=True)
for name, content in p["files"].items():
    dest = out / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(content)
zip_path = shutil.make_archive(p["project_name"], "zip", out.parent)
print("Created:", os.path.relpath(zip_path))
# Colab: from google.colab import files; files.download(zip_path)

Created: SmartSpend.zip


## Wrapping up

The app remembers what it built because every stage reads and writes the same project state: requirements, your answers, the architecture, the files, test results, explanations and your learning progress. Explaining, fixing and learning all start from that state, not from a fresh prompt.

So the loop is: generate, review, fix, verify, then learn from the result.